# CR-1a 段3 本番のリクエストの一覧づくり（CPU・API なし）

本番（G2）で送るリクエストの一覧を、G1 の非公開の土台の道から凍結した決まりどおりに作り、**ハッシュだけ**を書きます。

- Claude の API は使いません（費用 $0）。`ANTHROPIC_API_KEY` も `HF_TOKEN` も要りません。要るのは `GITHUB_TOKEN` だけ。
- 問題文・応答はどこにも表示しません。一覧には番号とハッシュだけが入ります。
- セル1 → セル2 を実行（全部で 2〜3 分）。結果は Drive の `DECISION_POINTS/cr1a/G2/` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, notifications.  No API key; Hugging Face is read without any token.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"     # never look for HF_TOKEN (the dataset is public)

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "926c25fae380c3421a3af62aa1a28ca830162769"
G1_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
G1_ITEMS_SHA = "bd42639245cb234f86e0bcd97092db67ae2842c9f1e48e123a9ee1b484c931b5"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", G1_HEAD, head]).returncode == 0, "凍結コミットが祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import cr1_run as R
from pork import cr1_requests as RQ
from pork import cr1_analysis as A
R.check_study(STUDY)
print("study:", STUDY, "| exact head:", head)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1a 段3 本番リクエスト一覧づくり（CPU・API なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1a lock3 manifest', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — rebuild everything from the frozen rules and the PRIVATE G1 checkpoint; write hashes only.
from pork import cr1_items as CI
rows, DATA_ID = R.load_rows_public()
chain = CI.chain_and_candidates(rows)
del rows
assert CI.candidates_sha256(chain) == R.CANDIDATES_SHA256, "候補の一覧が段2 と違います。止めます。"
G1 = Path("/content/drive/MyDrive") / R.paths(STUDY)["dir"]
ck = json.loads((G1 / "CR1A_G1_checkpoint_PRIVATE.json").read_text())
want = R.header(STUDY, G1_HEAD, R.CANDIDATES_SHA256, A.MODEL, RQ.SDK_VERSION)
assert ck["header"] == want, "G1 の保存ファイルの見出しが違います。止めます。"
R.validate(ck, chain)
assert not ck["pending"] and not ck["partial"]
g1 = R.summarize_g1(ck)                       # recomputed from the checkpoint, not read from the result file
assert g1["verdict"] == "GO" and g1["final_items_ids_sha256"] == G1_ITEMS_SHA, "G1b の結果が記録と違います。止めます。"
saved = json.loads((G1 / "CR1A_G1.json").read_text())
assert saved["final_items_ids_sha256"] == G1_ITEMS_SHA and saved["dataset"] == DATA_ID
manifest, summary = R.build_g2_manifest(g1["final_items"], ck["items"], chain, STUDY)
again = R.build_g2_manifest(g1["final_items"], ck["items"], chain, STUDY)[1]["manifest_sha256"]
assert again == summary["manifest_sha256"], "一覧が 2 回で一致しません。止めます。"
summary.update(code_head=head, g1_code_head=G1_HEAD, dataset=DATA_ID, g1b_estimate_usd=g1["G1b"]["estimate"]["usd"],
               dropped_templates=g1["G1b"]["dropped_templates"])
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a/G2")
OUT.mkdir(parents=True, exist_ok=True)
R.drive_smoke(OUT)
R._atomic_write(OUT / "CR1A_G2_manifest.json", manifest)
R._atomic_write(OUT / "CR1A_LOCK3_summary.json", summary)
assert R.canonical_sha256(json.loads((OUT / "CR1A_G2_manifest.json").read_text())) == summary["manifest_sha256"]
print("問題の数:", summary["n_items"], "| 切れ目の行:", summary["n_rows"], "| リクエスト:", summary["n_requests"])
print("一覧の sha256:", summary["manifest_sha256"])
notify('終わりました：本番のリクエスト一覧（ハッシュだけ）を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()